In [387]:
import pandas as pd
import numpy as np
import json, csv, os
from typing import List, Dict, AnyStr

In [388]:
df = pd.read_csv("datasets\\googleplaystore.csv")
review_df = pd.read_csv("datasets\\googleplaystore_user_reviews.csv")

df.shape, review_df.shape

((10841, 13), (64295, 5))

In [389]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


### Data Cleaning
- Let's try to target Null, NaN and empty values

In [390]:
pd.DataFrame({0: df.isna().sum(),1: df.isnull().sum()})

# we can clean rating when some questions can be asked or else we can leverage that rows for other features
# Better to ignore versions of apps here, we can even drop those columns

,0,1
App,0,0
Category,0,0
Rating,1474,1474
Reviews,0,0
Size,0,0
Installs,0,0
Type,1,1
Price,0,0
Content Rating,1,1
Genres,0,0


In [391]:
# Dropping the version column because No usecase in my mid for those column as such.
df.drop(["Current Ver", "Android Ver"], axis=1, inplace=True)

### Data Processing:

In [392]:
# We can strip App with "" as reviewed:
df.App.value_counts().to_csv('test101.csv')
df.loc[:, 'App'] = df.App.str.strip('"')

In [393]:
# Let's convert this into Category dtype:
df.Category.value_counts().sort_index().head()
df.Category = df.Category.astype('category')

In [394]:
# Invalid data found as per research(Values are swap here):
df[~df.Reviews.str.isnumeric()].index

i = 10472
df.Rating = df.Rating.astype(str)
df.iloc[i, 2:] = df.iloc[i, 1:-1].values
df.iloc[i, 1] = 'FAMILY'

df.Rating = df.Rating.astype('Float32')
# Due to column values are placed incorrectly, We can use iloc to swap values to the next value easily, before that List values and datatype must be matched

In [395]:
df.iloc[i]

App               Life Made WI-Fi Touchscreen Photo Frame
Category                                           FAMILY
Rating                                                1.9
Reviews                                              19.0
Size                                                 3.0M
Installs                                           1,000+
Type                                                 Free
Price                                                   0
Content Rating                                   Everyone
Genres                                                NaN
Last Updated                            February 11, 2018
Name: 10472, dtype: object

In [396]:
df.Reviews = df.Reviews.astype('Float32')

In [397]:
# df[~df.Size.str.contains(pat='M|k') & ~(df.Size == 'Varies with device')]
df.Size = df.Size.str.replace(pat={r'\dM$': r" MB", r'\dk$': " KB"}, regex=True)

In [398]:
df.Installs = df.Installs.str.replace({',': "", '+': ""})

In [399]:
df.iloc[9148, 6] = 'Free'
df.Type = df.Type.astype('category')

In [400]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   App             10841 non-null  str     
 1   Category        10841 non-null  category
 2   Rating          9367 non-null   Float32 
 3   Reviews         10841 non-null  Float32 
 4   Size            10841 non-null  str     
 5   Installs        10841 non-null  str     
 6   Type            10841 non-null  category
 7   Price           10841 non-null  str     
 8   Content Rating  10841 non-null  str     
 9   Genres          10840 non-null  str     
 10  Last Updated    10841 non-null  str     
dtypes: Float32(2), category(2), str(7)
memory usage: 721.4 KB


In [401]:
# Random sampling for Price column so that we can observe values
vals = []
for _ in range(10):
    vals.extend(df.Price.sample(n=10).values)

vals = set(vals)
vals

{'$1.49', '$1.99', '$2.99', '$29.99', '$5.99', '0'}

In [402]:
# Replacing $ and coverting Price into float32:
df.loc[:, 'Price'] = df.Price.str.replace({'$': ''})

df.Price = df.Price.astype('float32')

##### Cleaning and processing Content Rating:

In [403]:
df['Content Rating'].value_counts()

Content Rating
Everyone           8715
Teen               1208
Mature 17+          499
Everyone 10+        414
Adults only 18+       3
Unrated               2
Name: count, dtype: int64

In [404]:
# it seems we can better to convert column into category here:
df['Content Rating'] = df['Content Rating'].astype('category')
df['Content Rating'].cat.categories

Index(['Adults only 18+', 'Everyone', 'Everyone 10+', 'Mature 17+', 'Teen',
       'Unrated'],
      dtype='str')

##### Genres Cleaning and Processing

In [405]:
df.Genres.value_counts()
# Multiple genres are sepearted by semi colon here, let's clean and process these out

Genres
Tools                       842
Entertainment               623
Education                   549
Medical                     463
Business                    460
                           ... 
Role Playing;Brain Games      1
Strategy;Education            1
Racing;Pretend Play           1
Communication;Creativity      1
Strategy;Creativity           1
Name: count, Length: 119, dtype: int64

In [406]:
# Replacing Nan Value
df.iloc[10472, 9] = 'Educational'

df.loc[df.Genres.str.contains(';'), 'Genres']

1              Art & Design;Pretend Play
4                Art & Design;Creativity
9                Art & Design;Creativity
23       Art & Design;Action & Adventure
26               Art & Design;Creativity
                      ...               
10438            Art & Design;Creativity
10493                Casual;Pretend Play
10502          Racing;Action & Adventure
10506                Casual;Pretend Play
10526                Strategy;Creativity
Name: Genres, Length: 498, dtype: str

##### Approach:
 - Collect all genres and Filter out based on requirement.

In [407]:
# Overview Genres:
genres = df.Genres.str.split(';', expand=True)
genres.values

genres = [item for each in genres.values for item in each]

genres = pd.Series(genres, name='unique_genres')
genres.dropna(inplace=True)
genres.drop_duplicates(inplace=True)
genres.reset_index(drop=True, inplace=True)
print(f"Unique Genres found: {len(genres)}")
genres

Unique Genres found: 53


0                Art & Design
1                Pretend Play
2                  Creativity
3          Action & Adventure
4             Auto & Vehicles
5                      Beauty
6           Books & Reference
7                    Business
8                      Comics
9               Communication
10                     Dating
11                  Education
12              Music & Video
13                Brain Games
14              Entertainment
15                     Events
16                    Finance
17               Food & Drink
18           Health & Fitness
19               House & Home
20           Libraries & Demo
21                  Lifestyle
22                  Adventure
23                     Arcade
24                     Casual
25                       Card
26                     Action
27                   Strategy
28                     Puzzle
29                     Sports
30                      Music
31                       Word
32                     Racing
33        

In [408]:
# Filter out based on specific need:

input_genre = ['Art & Design', 'Educational']
df[df.Genres.str.contains(pat='|'.join(input_genre), regex=True)]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,1 MB,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018"
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,1 MB,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018"
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8. MB,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018"
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,2 MB,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018"
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2. MB,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018"
...,...,...,...,...,...,...,...,...,...,...,...
9890,TAXLANDIA,FAMILY,1.9,141.0,6 MB,1000,Free,0.0,Everyone,Educational,"March 6, 2018"
9926,Going Abroad,FAMILY,3.0,606.0,4 MB,50000,Free,0.0,Everyone 10+,Educational,"August 16, 2016"
10394,PBS KIDS Games,FAMILY,4.3,12919.0,9 MB,1000000,Free,0.0,Everyone,Educational;Education,"June 6, 2018"
10438,Dolphin and fish coloring book,FAMILY,3.9,2249.0,Varies with device,500000,Free,0.0,Everyone,Art & Design;Creativity,"May 15, 2018"


In [409]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], format='%B %d, %Y')
df.Year = df['Last Updated'].dt.year
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             10841 non-null  str           
 1   Category        10841 non-null  category      
 2   Rating          9367 non-null   Float32       
 3   Reviews         10841 non-null  Float32       
 4   Size            10841 non-null  str           
 5   Installs        10841 non-null  str           
 6   Type            10841 non-null  category      
 7   Price           10841 non-null  float32       
 8   Content Rating  10841 non-null  category      
 9   Genres          10841 non-null  str           
 10  Last Updated    10841 non-null  datetime64[us]
dtypes: Float32(2), category(3), datetime64[us](1), float32(1), str(4)
memory usage: 605.0 KB


C:\Users\gaurav.009\AppData\Local\Temp\ipykernel_12452\2539312853.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.Year = df['Last Updated'].dt.year


In [410]:
# Dropping duplicates: (Multiple apps are found with same values apart from review count, Better to drop those out)
df.drop_duplicates(subset=['App', 'Size', 'Type', 'Last Updated'], inplace=True)

## Our Dataset looks super clean now ✨🔰, 
### Let's solve interesting questions:

Analytics:
1. Get recent apps which has above 4.5 rating
2. Get Free Apps for top 3 Downloads
3. Get most popular Genres of all time
4. What are the top 5 apps with the highest ratings?
5. How many unique categories exist in the dataset?
6. Which app has the largest number of reviews?
7. Calculate the average app size for each category.
8. Find the most reviewed app in each category.
9. What is the average rating per category?
10. Which category has the highest total installs?
11. Rank apps by number of installs.
12. Find the top 10 apps by reviews.
13. Sort apps by last updated date to see the most recent ones.
14. Compare free vs paid apps in terms of average rating and installs.
15. Find the distribution of content ratings (Everyone, Teen, etc.).
16. Which genre has the highest average rating?
17. Identify if there’s any correlation between app size and rating.

Visualization:
1. Bar chart of average rating per category.
2. Histogram of app sizes.
3. Pie chart of content ratings distribution.
4. Scatter plot of reviews vs installs.





#### Analytics:

In [411]:
# 1:
df.loc[(df.Year >= 2017) & (df.Rating >= 4.5), 'App']

# Data has last updated app upto 2018, hence I provide 2017 here

2        U Launcher Lite – FREE Live Cool Themes, Hide ...
3                                    Sketch - Draw & Paint
9                            Kids Paint Free - Drawing Fun
13                                   Mandala Coloring Book
16            Photo Designer - Write your name with shapes
                               ...                        
10810                                    Fr Lupupa Sermons
10820                                      Fr. Daoud Lamei
10836                                     Sya9a Maroc - FR
10837                     Fr. Mike Schmitz Audio Teachings
10840        iHoroscope - 2018 Daily Horoscope & Astrology
Name: App, Length: 2304, dtype: str

In [412]:
# 2:
top_5_installs = df.Installs.isin(df.Installs.value_counts().index.astype('int').sort_values(ascending=False).astype('str')[:5].to_list())
df.loc[(df.Type == 'Free') & (top_5_installs)].sort_values(by=['Installs', 'Rating'], ascending=[False, False])

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated
4005,Clean Master- Space Cleaner & Antivirus,TOOLS,4.7,42916528.0,Varies with device,500000000,Free,0.0,Everyone,Tools,2018-08-03
7536,"Security Master - Antivirus, VPN, AppLock, Boo...",TOOLS,4.7,24901000.0,Varies with device,500000000,Free,0.0,Everyone,Tools,2018-08-04
371,Google Duo - High Quality Video Calls,COMMUNICATION,4.6,2083237.0,Varies with device,500000000,Free,0.0,Everyone,Communication,2018-07-31
3255,SHAREit - Transfer & Share,TOOLS,4.6,7790693.0,1 MB,500000000,Free,0.0,Everyone,Tools,2018-07-30
378,UC Browser - Fast Download Private & Secure,COMMUNICATION,4.5,17712922.0,4 MB,500000000,Free,0.0,Teen,Communication,2018-08-02
...,...,...,...,...,...,...,...,...,...,...,...
6327,AfreecaTV,VIDEO_PLAYERS,3.3,381023.0,5 MB,10000000,Free,0.0,Everyone,Video Players & Editors,2018-07-26
8316,Scratch Logo Quiz. Challenging brain puzzle,GAME,3.3,152102.0,1 MB,10000000,Free,0.0,Teen,Trivia,2018-06-30
3262,Clear,TOOLS,3.1,24151.0,1 MB,10000000,Free,0.0,Everyone,Tools,2018-08-06
3483,HTC Mail,PRODUCTIVITY,3.1,6572.0,Varies with device,10000000,Free,0.0,Everyone,Productivity,2018-05-29


In [413]:
# 3:
# Popularity measured based on Rating, Positive Reviews and Installs as per given data:

df.loc[(df.Rating > 4.7) & (top_5_installs)].groupby(['Rating', 'Genres']).count()['App'].sort_values(ascending=False).sort_index(level=[0])

Rating  Genres                   
4.8     Arcade                       2
        Books & Reference            1
        Casual                       1
        Entertainment;Brain Games    1
        Health & Fitness             6
        Role Playing                 1
        Simulation                   1
        Social                       2
        Sports                       1
        Strategy                     1
        Tools                        1
        Video Players & Editors      3
        Word                         1
4.9     Books & Reference            1
        Health & Fitness             1
Name: App, dtype: int64

In [414]:
# 4
df.loc[(df.Rating >= 4.7), ['App', 'Rating']].drop_duplicates()

,App,Rating
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",4.7
9,Kids Paint Free - Drawing Fun,4.7
16,Photo Designer - Write your name with shapes,4.7
22,Superheroes Wallpapers | 4K Backgrounds,4.7
24,HD Mickey Minnie Wallpapers,4.7
...,...,...
10809,Castle Clash: RPG War and Strategy FR,4.7
10810,Fr Lupupa Sermons,4.8
10820,Fr. Daoud Lamei,5.0
10833,Chemin (fr),4.8


In [415]:
# 5
# Already found in previous data cleaning process
genres[:10]

0          Art & Design
1          Pretend Play
2            Creativity
3    Action & Adventure
4       Auto & Vehicles
5                Beauty
6     Books & Reference
7              Business
8                Comics
9         Communication
Name: unique_genres, dtype: str

In [439]:
# 6
df.loc[:, ['App', 'Reviews']].nlargest(1, columns=['Reviews']).drop_duplicates().round(7).reset_index(drop=True)

,App,Reviews
0,Facebook,78158304.0


In [437]:
# 7:
# Since app size available in MB/KB/Varies with device options, we need filter values accordingly
print("Average App size:",
      float(df.Size[df.Size.str.contains(r'MB|KB')].str.strip(r'KB|MB').astype('float32').mean()))

Average App size: 5.093380451202393


In [438]:
# 8:
df.Reviews.nlargest(10)

2544    78158304.0
336     69119312.0
2545    66577312.0
335     56642848.0
1670    44891724.0
4005    42916528.0
1654    27722264.0
3665    25655304.0
7536    24901000.0
1660    23133508.0
Name: Reviews, dtype: Float32

In [441]:
# 8 (Same Logic from Question 6's Answer):
df.loc[:, ['App', 'Reviews']].nlargest(10, columns=['Reviews']).drop_duplicates().reset_index(drop=True)

,App,Reviews
0,Facebook,78158304.0
1,WhatsApp Messenger,69119312.0
2,Instagram,66577312.0
3,Messenger – Text and Video Chat for Free,56642848.0
4,Clash of Clans,44891724.0
5,Clean Master- Space Cleaner & Antivirus,42916528.0
6,Subway Surfers,27722264.0
7,YouTube,25655304.0
8,"Security Master - Antivirus, VPN, AppLock, Boo...",24901000.0
9,Clash Royale,23133508.0


In [454]:
# 9
df.groupby(['Category'])['Rating'].mean().sort_values(ascending=False).reset_index(name='Average_Rating')

,Category,Average_Rating
0,EVENTS,4.435555
1,EDUCATION,4.364407
2,ART_AND_DESIGN,4.357377
3,BOOKS_AND_REFERENCE,4.34497
4,PERSONALIZATION,4.332215
5,PARENTING,4.3
6,BEAUTY,4.278571
7,SOCIAL,4.25
8,GAME,4.248363
9,WEATHER,4.245205


In [459]:
# 10:
df.groupby('Category')['Installs'].max().reset_index(name='Maximum_Installs').sort_values(by=['Category', 'Maximum_Installs'], ascending=[True, False])

,Category,Maximum_Installs
0,ART_AND_DESIGN,50000000
1,AUTO_AND_VEHICLES,5000000
2,BEAUTY,5000000
3,BOOKS_AND_REFERENCE,5000000
4,BUSINESS,50000000
5,COMICS,5000000
6,COMMUNICATION,500000000
7,DATING,5000000
8,EDUCATION,5000000
9,ENTERTAINMENT,50000000


In [ ]:
# 11:


10. Which category has the highest total installs?
11. Rank apps by number of installs.
12. Find the top 10 apps by reviews.
13. Sort apps by last updated date to see the most recent ones.
14. Compare free vs paid apps in terms of average rating and installs.
15. Find the distribution of content ratings (Everyone, Teen, etc.).
16. Which genre has the highest average rating?
17. Identify if there’s any correlation between app size and rating.

In [427]:
# df.App[df.App.duplicated()]
df[df.App == 'Solitaire']

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated
1656,Solitaire,GAME,4.7,254258.0,2 MB,10000000,Free,0.0,Everyone,Card,2018-08-01
1973,Solitaire,GAME,4.7,154264.0,1 MB,10000000,Free,0.0,Everyone,Card,2018-06-08
2024,Solitaire,FAMILY,4.4,685.0,2 MB,100000,Free,0.0,Everyone,Card;Brain Games,2018-07-16


#### Visualization: